# Buzzlytics — Build the dataset zip (RUN ONCE)

Builds the unified 3-class `bee_dataset.zip` from the two public sources and saves it to
your Google Drive. After this, you never run it again — just paste the zip's share link into
`colab_train.ipynb` and train.

- **VnPollenBee** (HUST) — pulled via the Drive API (the folder has >50 files, so gdown can't).
- **VarroaDataset** (Zenodo 4085044) — downloaded directly.

## Run
1. Runtime → Run all. Approve the two Google popups (API auth + Drive mount).
2. At the end: in Drive, right-click `bee_dataset.zip` → Share → **Anyone with the link** → Copy link.
3. Paste that link into `colab_train.ipynb` → `DATASET_URL`.

In [ ]:
# === 0. Config ===
REPO_URL = "https://github.com/okayxsh/Buzzlytics---CSCI435.git"
REPO_DIR = "/content/Buzzlytics---CSCI435"
VPB_FOLDER_ID = "1fdEcu7CNmEkVAamu9wh_Ppw_-uW3VNY1"   # VnPollenBee Drive folder
VARROA_LIMIT = 3000   # cap redundant varroa crops (class-balanced). None = all 13.5k
OUT_ZIP = "/content/drive/MyDrive/bee_dataset.zip"

In [ ]:
# === 1. Install deps + clone repo (for the converters) ===
!pip -q install gdown pyyaml
import os
if not os.path.isdir(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull --ff-only
%cd {REPO_DIR}

In [ ]:
# === 2. Pull VnPollenBee LabelMe JSONs via the Drive API (parallel) ===
import io, shutil, threading
from concurrent.futures import ThreadPoolExecutor
from google.colab import auth
from googleapiclient.discovery import build as gbuild
from googleapiclient.http import MediaIoBaseDownload
auth.authenticate_user()
VPB_DIR = "/content/vnpollenbee"

def _api():
    return gbuild("drive", "v3", cache_discovery=False)

def _children(api, fid):
    items, tok = [], None
    while True:
        r = api.files().list(q=f"'{fid}' in parents and trashed=false",
            fields="nextPageToken, files(id, name, mimeType)", pageSize=1000,
            pageToken=tok, supportsAllDrives=True, includeItemsFromAllDrives=True).execute()
        items += r["files"]; tok = r.get("nextPageToken")
        if not tok: break
    return items

def _walk(api, fid, dest):
    os.makedirs(dest, exist_ok=True)
    jobs = []
    for f in _children(api, fid):
        if f["mimeType"] == "application/vnd.google-apps.folder":
            jobs += _walk(api, f["id"], os.path.join(dest, f["name"]))
        elif f["name"].lower().endswith(".json"):
            jobs.append((f["id"], os.path.join(dest, f["name"])))
    return jobs

_tl = threading.local()
def _get(job):
    if not hasattr(_tl, "api"): _tl.api = _api()
    fid, path = job
    dl = MediaIoBaseDownload(io.FileIO(path, "wb"), _tl.api.files().get_media(fileId=fid))
    done = False
    while not done: _, done = dl.next_chunk()

shutil.rmtree(VPB_DIR, ignore_errors=True)
print("Enumerating VnPollenBee folder...")
jobs = _walk(_api(), VPB_FOLDER_ID, VPB_DIR)
print(f"  {len(jobs)} JSONs -> downloading in parallel...")
with ThreadPoolExecutor(max_workers=16) as ex:
    list(ex.map(_get, jobs))
print("  done")

In [ ]:
# === 3. Build the dataset INLINE (errors show here directly) ===
import sys
from pathlib import Path
if REPO_DIR not in sys.path: sys.path.insert(0, REPO_DIR)
from datasets.prepare_dataset import (
    prepare_vnpollenbee, prepare_varroa, write_data_yaml, _ensure_dirs)

RAW = Path("datasets/raw"); OUT = Path("datasets/data")
shutil.rmtree(OUT, ignore_errors=True)
RAW.mkdir(parents=True, exist_ok=True); _ensure_dirs(OUT)

n_vpb = prepare_vnpollenbee(None, RAW, OUT, local_dir=VPB_DIR)
n_var = prepare_varroa(RAW, OUT, limit=VARROA_LIMIT)
write_data_yaml(OUT)
print(f"\nBuilt {n_vpb + n_var} images (VnPollenBee {n_vpb}, Varroa {n_var}).")

In [ ]:
# === 4. Sanity: class distribution (0=bee 1=pollen 2=varroa) ===
from collections import Counter
c = Counter()
for txt in Path("datasets/data").rglob("labels/*.txt"):
    for line in txt.read_text().splitlines():
        if line.strip(): c[int(line.split()[0])] += 1
print("instances per class:", dict(sorted(c.items())))
for s in ["train", "valid", "test"]:
    d = Path("datasets/data") / s / "images"
    print(f"{s}: {len(list(d.glob('*'))) if d.is_dir() else 0} images")

In [ ]:
# === 5. Zip + save to Google Drive ===
from google.colab import drive
drive.mount('/content/drive')
shutil.make_archive(OUT_ZIP[:-4], 'zip', root_dir='datasets', base_dir='data')
sz = os.path.getsize(OUT_ZIP) / 1e6
print(f"Saved {OUT_ZIP}  ({sz:.0f} MB)")
print("\nNext: in Drive, right-click bee_dataset.zip -> Share -> Anyone with the link")
print("-> Copy link -> paste into colab_train.ipynb DATASET_URL.")